In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import pickle


In [3]:
### Load model, scaler, onehot encoder
model = load_model('model.h5')

# Load scalar
with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

In [23]:
input={
    'CreditScore': 600,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' : 1,
    'EstimatedSalary': 50000
}
input_df = pd.DataFrame([input])


In [24]:
input_df["Gender"] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [25]:

geo_col = onehot_encoder_geo.transform(input_df[['Geography']])
geo_col

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1 stored elements and shape (1, 3)>

In [26]:

df = pd.DataFrame(geo_col.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
data = pd.concat([input_df.drop(['Geography'], axis=1), df], axis=1)

data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [28]:
inp_scaled = scaler.transform(data)

In [ ]:
prediction=model.predict(inp_scaled)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


array([[0.0173247]], dtype=float32)

In [30]:
prediction_probability = prediction[0][0]
if prediction_probability >= 0.5:
    print('The customer will leave')
else:
    print("The customer won't leave")

The customer won't leave
